# DUBLIN PUBLIC TRANSPORT FARE COMPLIANCE & PASSENGER JOURNEY ANALYTICS

## NOTEBOOK: 02_gtfs_data_ingestion

PURPOSE: Prepare the Databricks environment for ingestion and processing of National Transport Authority GTFS data.

 DATA ARCHITECTURE:

NTA GTFS Files
      |
      v
Bronze GTFS Tables
      |
      v
Silver GTFS Tables
      |
      v
Passenger Journey + GTFS Integration
      |
      v
Gold Analytics Layer

IMPORTANT: GTFS reference data = real public transport data
Passenger journey/compliance data = synthetic project data

In [0]:


from pyspark.sql import functions as F
from pyspark.sql.functions import col, trim, when, to_date

print("=" * 65)
print("DUBLIN TRANSPORT GTFS INGESTION PIPELINE")
print("=" * 65)

print("\nPySpark environment loaded successfully.")

# Set project catalog and schema
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

print("Catalog: workspace")
print("Schema: default")

print("\nGTFS ingestion environment ready.")

DUBLIN TRANSPORT GTFS INGESTION PIPELINE

PySpark environment loaded successfully.
Catalog: workspace
Schema: default

GTFS ingestion environment ready.


## STEP 2: VERIFY RAW GTFS FILES
Dublin Public Transport Fare Compliance & Passenger Journey Analytics

In [0]:

print("=" * 70)
print("RAW GTFS FILE VERIFICATION")
print("=" * 70)

# Location of the raw NTA GTFS files in the Unity Catalog volume
gtfs_raw_path = "/Volumes/workspace/default/dublin_transport_gtfs/raw_gtfs"

print(f"\nGTFS raw data path:")
print(gtfs_raw_path)

# List files stored in the raw GTFS directory
gtfs_files = dbutils.fs.ls(gtfs_raw_path)

print("\nFILES FOUND")
print("-" * 70)

for file in sorted(gtfs_files, key=lambda x: x.name):
    print(f"{file.name:<25} {file.size:>15,} bytes")

# Count uploaded files
file_count = len(gtfs_files)

print("\n" + "=" * 70)
print("GTFS FILE SUMMARY")
print("=" * 70)
print(f"Total GTFS files found: {file_count}")

# Files expected from our upload
expected_files = {
    "agency.txt",
    "calendar.txt",
    "calendar_dates.txt",
    "feed_info.txt",
    "routes.txt",
    "shapes.txt",
    "stop_times.txt",
    "stops.txt",
    "translations.txt",
    "trips.txt"
}

actual_files = {file.name for file in gtfs_files}

missing_files = expected_files - actual_files
unexpected_files = actual_files - expected_files

print(f"Expected files: {len(expected_files)}")
print(f"Missing files: {len(missing_files)}")
print(f"Unexpected files: {len(unexpected_files)}")

if missing_files:
    print("\nMissing:")
    for file in sorted(missing_files):
        print(f" - {file}")

if unexpected_files:
    print("\nUnexpected:")
    for file in sorted(unexpected_files):
        print(f" - {file}")

if file_count == 10 and len(missing_files) == 0:
    print("\nGTFS RAW FILE VALIDATION STATUS: PASS")
else:
    print("\nGTFS RAW FILE VALIDATION STATUS: REVIEW REQUIRED")

print("=" * 70)

RAW GTFS FILE VERIFICATION

GTFS raw data path:
/Volumes/workspace/default/dublin_transport_gtfs/raw_gtfs

FILES FOUND
----------------------------------------------------------------------
agency.txt                          7,335 bytes
calendar.txt                       10,533 bytes
calendar_dates.txt                 28,434 bytes
feed_info.txt                         266 bytes
routes.txt                         41,223 bytes
shapes.txt                    294,766,625 bytes
stop_times.txt                373,052,021 bytes
stops.txt                         862,351 bytes
translations.txt                  699,982 bytes
trips.txt                      15,999,647 bytes

GTFS FILE SUMMARY
Total GTFS files found: 10
Expected files: 10
Missing files: 0
Unexpected files: 0

GTFS RAW FILE VALIDATION STATUS: PASS


## STEP 3: LOAD RAW GTFS FILES WITH PYSPARK
Dublin Public Transport Fare Compliance & Passenger Journey Analytics

In [0]:
print("=" * 70)
print("LOADING RAW GTFS DATASETS")
print("=" * 70)

gtfs_raw_path = "/Volumes/workspace/default/dublin_transport_gtfs/raw_gtfs"

# GTFS datasets to load

gtfs_dataset_names = [
    "agency",
    "calendar",
    "calendar_dates",
    "feed_info",
    "routes",
    "shapes",
    "stop_times",
    "stops",
    "translations",
    "trips"
]

# Dictionary used to store each Spark DataFrame
gtfs_dfs = {}

#Load each GTFS text file as CSV

for dataset_name in gtfs_dataset_names:

    file_path = f"{gtfs_raw_path}/{dataset_name}.txt"

    print(f"\nLoading: {dataset_name}.txt")

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("mode", "PERMISSIVE")
        .csv(file_path)
    )

    gtfs_dfs[dataset_name] = df

    print(f"Loaded successfully: {dataset_name}")

print("\n" + "=" * 70)
print("ALL GTFS FILES LOADED INTO PYSPARK")
print("=" * 70)

# Create easy-to-use DataFrame variables

agency = gtfs_dfs["agency"]
calendar = gtfs_dfs["calendar"]
calendar_dates = gtfs_dfs["calendar_dates"]
feed_info = gtfs_dfs["feed_info"]
routes = gtfs_dfs["routes"]
shapes = gtfs_dfs["shapes"]
stop_times = gtfs_dfs["stop_times"]
stops = gtfs_dfs["stops"]
translations = gtfs_dfs["translations"]
trips = gtfs_dfs["trips"]

print("\nPySpark DataFrames created:")
print(" - agency")
print(" - calendar")
print(" - calendar_dates")
print(" - feed_info")
print(" - routes")
print(" - shapes")
print(" - stop_times")
print(" - stops")
print(" - translations")
print(" - trips")

# Inspect schema dimensions

print("\n" + "=" * 70)
print("GTFS DATASET STRUCTURE")
print("=" * 70)

structure_results = []

for dataset_name, df in gtfs_dfs.items():

    column_count = len(df.columns)

    structure_results.append(
        (dataset_name, column_count)
    )

structure_df = spark.createDataFrame(
    structure_results,
    ["dataset", "column_count"]
)

display(
    structure_df.orderBy("dataset")
)

print("\nGTFS PySpark loading stage completed successfully.")

LOADING RAW GTFS DATASETS

Loading: agency.txt
Loaded successfully: agency

Loading: calendar.txt
Loaded successfully: calendar

Loading: calendar_dates.txt
Loaded successfully: calendar_dates

Loading: feed_info.txt
Loaded successfully: feed_info

Loading: routes.txt
Loaded successfully: routes

Loading: shapes.txt
Loaded successfully: shapes

Loading: stop_times.txt
Loaded successfully: stop_times

Loading: stops.txt
Loaded successfully: stops

Loading: translations.txt
Loaded successfully: translations

Loading: trips.txt
Loaded successfully: trips

ALL GTFS FILES LOADED INTO PYSPARK

PySpark DataFrames created:
 - agency
 - calendar
 - calendar_dates
 - feed_info
 - routes
 - shapes
 - stop_times
 - stops
 - translations
 - trips

GTFS DATASET STRUCTURE


dataset,column_count
agency,4
calendar,10
calendar_dates,3
feed_info,7
routes,9
shapes,5
stop_times,9
stops,11
translations,6
trips,8



GTFS PySpark loading stage completed successfully.


## STEP 4: GTFS SCHEMA AND DATA QUALITY VALIDATION
Dublin Public Transport Fare Compliance & Passenger Journey Analytics

In [0]:
print("=" * 70)
print("GTFS SCHEMA AND DATA QUALITY VALIDATION")
print("=" * 70)

# Expected columns for each GTFS dataset

expected_columns = {
    "agency": [
        "agency_id",
        "agency_name",
        "agency_url",
        "agency_timezone"
    ],

    "calendar": [
        "service_id",
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday",
        "saturday",
        "sunday",
        "start_date",
        "end_date"
    ],

    "calendar_dates": [
        "service_id",
        "date",
        "exception_type"
    ],

    "feed_info": [
        "feed_publisher_name",
        "feed_publisher_url",
        "feed_lang",
        "feed_start_date",
        "feed_end_date",
        "feed_version",
        "feed_contact_email"
    ],

    "routes": [
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name",
        "route_desc",
        "route_type",
        "route_url",
        "route_color",
        "route_text_color"
    ],

    "shapes": [
        "shape_id",
        "shape_pt_lat",
        "shape_pt_lon",
        "shape_pt_sequence",
        "shape_dist_traveled"
    ],

    "stop_times": [
        "trip_id",
        "arrival_time",
        "departure_time",
        "stop_id",
        "stop_sequence",
        "stop_headsign",
        "pickup_type",
        "drop_off_type",
        "timepoint"
    ],

    "stops": [
        "stop_id",
        "stop_code",
        "stop_name",
        "stop_desc",
        "stop_lat",
        "stop_lon",
        "zone_id",
        "stop_url",
        "location_type",
        "parent_station",
        "wheelchair_boarding"
    ],

    "translations": [
        "table_name",
        "field_name",
        "language",
        "translation",
        "record_id",
        "field_value"
    ],

    "trips": [
        "route_id",
        "service_id",
        "trip_id",
        "trip_headsign",
        "trip_short_name",
        "direction_id",
        "block_id",
        "shape_id"
    ]
}

# Validate columns

print("\n" + "=" * 70)
print("COLUMN VALIDATION")
print("=" * 70)

schema_validation_results = []

for dataset_name, expected in expected_columns.items():

    df = gtfs_dfs[dataset_name]

    actual = df.columns

    missing = sorted(list(set(expected) - set(actual)))
    unexpected = sorted(list(set(actual) - set(expected)))

    status = (
        "PASS"
        if len(missing) == 0 and len(unexpected) == 0
        else "REVIEW"
    )

    schema_validation_results.append(
        (
            dataset_name,
            len(expected),
            len(actual),
            ", ".join(missing) if missing else "None",
            ", ".join(unexpected) if unexpected else "None",
            status
        )
    )

schema_validation_df = spark.createDataFrame(
    schema_validation_results,
    [
        "dataset",
        "expected_columns",
        "actual_columns",
        "missing_columns",
        "unexpected_columns",
        "status"
    ]
)

display(
    schema_validation_df.orderBy("dataset")
)

# Display inferred Spark data types

print("\n" + "=" * 70)
print("INFERRED DATA TYPES")
print("=" * 70)

datatype_results = []

for dataset_name, df in gtfs_dfs.items():

    for field in df.schema.fields:

        datatype_results.append(
            (
                dataset_name,
                field.name,
                field.dataType.simpleString()
            )
        )

datatype_df = spark.createDataFrame(
    datatype_results,
    [
        "dataset",
        "column_name",
        "spark_data_type"
    ]
)

display(
    datatype_df.orderBy(
        "dataset",
        "column_name"
    )
)

# Validate critical identifier columns

print("\n" + "=" * 70)
print("CRITICAL IDENTIFIER NULL CHECK")
print("=" * 70)

critical_identifiers = {
    "agency": ["agency_id"],
    "calendar": ["service_id"],
    "calendar_dates": ["service_id"],
    "routes": ["route_id"],
    "shapes": ["shape_id"],
    "stop_times": ["trip_id", "stop_id"],
    "stops": ["stop_id"],
    "trips": ["route_id", "service_id", "trip_id"]
}

identifier_results = []

for dataset_name, columns in critical_identifiers.items():

    df = gtfs_dfs[dataset_name]

    for column_name in columns:

        if column_name in df.columns:

            null_count = (
                df
                .filter(F.col(column_name).isNull())
                .limit(1)
                .count()
            )

            status = (
                "PASS"
                if null_count == 0
                else "REVIEW"
            )

            identifier_results.append(
                (
                    dataset_name,
                    column_name,
                    null_count,
                    status
                )
            )

        else:

            identifier_results.append(
                (
                    dataset_name,
                    column_name,
                    None,
                    "COLUMN MISSING"
                )
            )

identifier_validation_df = spark.createDataFrame(
    identifier_results,
    [
        "dataset",
        "identifier_column",
        "null_detected",
        "status"
    ]
)

display(
    identifier_validation_df.orderBy(
        "dataset",
        "identifier_column"
    )
)

# ------------------------------------------------------------
# 5. Duplicate primary identifier checks

# Only datasets where a single field should identify a record
# are checked here.

# stop_times, shapes and calendar_dates use combinations of
# columns, so they are not treated as single-column keys.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRIMARY IDENTIFIER DUPLICATE CHECK")
print("=" * 70)

primary_identifiers = {
    "routes": "route_id",
    "stops": "stop_id",
    "trips": "trip_id"
}

duplicate_results = []

for dataset_name, id_column in primary_identifiers.items():

    df = gtfs_dfs[dataset_name]

    duplicate_exists = (
        df
        .groupBy(id_column)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    status = (
        "PASS"
        if duplicate_exists == 0
        else "REVIEW"
    )

    duplicate_results.append(
        (
            dataset_name,
            id_column,
            duplicate_exists,
            status
        )
    )

duplicate_validation_df = spark.createDataFrame(
    duplicate_results,
    [
        "dataset",
        "identifier_column",
        "duplicate_detected",
        "status"
    ]
)

display(
    duplicate_validation_df.orderBy("dataset")
)

# Check key relationships between GTFS datasets

print("\n" + "=" * 70)
print("GTFS RELATIONSHIP VALIDATION")
print("=" * 70)

relationship_results = []

# routes.agency_id -> agency.agency_id

orphan_route_agency = (
    routes
    .select("agency_id")
    .filter(F.col("agency_id").isNotNull())
    .distinct()
    .join(
        agency.select("agency_id").distinct(),
        on="agency_id",
        how="left_anti"
    )
    .limit(1)
    .count()
)

relationship_results.append(
    (
        "routes.agency_id -> agency.agency_id",
        orphan_route_agency,
        "PASS" if orphan_route_agency == 0 else "REVIEW"
    )
)

# trips.route_id -> routes.route_id

orphan_trip_route = (
    trips
    .select("route_id")
    .filter(F.col("route_id").isNotNull())
    .distinct()
    .join(
        routes.select("route_id").distinct(),
        on="route_id",
        how="left_anti"
    )
    .limit(1)
    .count()
)

relationship_results.append(
    (
        "trips.route_id -> routes.route_id",
        orphan_trip_route,
        "PASS" if orphan_trip_route == 0 else "REVIEW"
    )
)

# stop_times.trip_id -> trips.trip_id

orphan_stop_time_trip = (
    stop_times
    .select("trip_id")
    .filter(F.col("trip_id").isNotNull())
    .distinct()
    .join(
        trips.select("trip_id").distinct(),
        on="trip_id",
        how="left_anti"
    )
    .limit(1)
    .count()
)

relationship_results.append(
    (
        "stop_times.trip_id -> trips.trip_id",
        orphan_stop_time_trip,
        "PASS" if orphan_stop_time_trip == 0 else "REVIEW"
    )
)

# stop_times.stop_id -> stops.stop_id

orphan_stop_time_stop = (
    stop_times
    .select("stop_id")
    .filter(F.col("stop_id").isNotNull())
    .distinct()
    .join(
        stops.select("stop_id").distinct(),
        on="stop_id",
        how="left_anti"
    )
    .limit(1)
    .count()
)

relationship_results.append(
    (
        "stop_times.stop_id -> stops.stop_id",
        orphan_stop_time_stop,
        "PASS" if orphan_stop_time_stop == 0 else "REVIEW"
    )
)

# trips.shape_id -> shapes.shape_id

orphan_trip_shape = (
    trips
    .select("shape_id")
    .filter(F.col("shape_id").isNotNull())
    .distinct()
    .join(
        shapes.select("shape_id").distinct(),
        on="shape_id",
        how="left_anti"
    )
    .limit(1)
    .count()
)

relationship_results.append(
    (
        "trips.shape_id -> shapes.shape_id",
        orphan_trip_shape,
        "PASS" if orphan_trip_shape == 0 else "REVIEW"
    )
)

relationship_validation_df = spark.createDataFrame(
    relationship_results,
    [
        "relationship",
        "orphan_detected",
        "status"
    ]
)

display(relationship_validation_df)

# Overall validation status

schema_failures = (
    schema_validation_df
    .filter(F.col("status") != "PASS")
    .count()
)

identifier_failures = (
    identifier_validation_df
    .filter(F.col("status") != "PASS")
    .count()
)

duplicate_failures = (
    duplicate_validation_df
    .filter(F.col("status") != "PASS")
    .count()
)

relationship_failures = (
    relationship_validation_df
    .filter(F.col("status") != "PASS")
    .count()
)

total_failures = (
    schema_failures
    + identifier_failures
    + duplicate_failures
    + relationship_failures
)

print("\n" + "=" * 70)
print("GTFS VALIDATION SUMMARY")
print("=" * 70)

print(f"Schema issues: {schema_failures}")
print(f"Identifier issues: {identifier_failures}")
print(f"Duplicate-key issues: {duplicate_failures}")
print(f"Relationship issues: {relationship_failures}")
print(f"Total validation issues: {total_failures}")

if total_failures == 0:

    print("\nGTFS DATA QUALITY VALIDATION STATUS: PASS")

else:

    print("\nGTFS DATA QUALITY VALIDATION STATUS: REVIEW REQUIRED")

print("=" * 70)

GTFS SCHEMA AND DATA QUALITY VALIDATION

COLUMN VALIDATION


dataset,expected_columns,actual_columns,missing_columns,unexpected_columns,status
agency,4,4,None,None,PASS
calendar,10,10,None,None,PASS
calendar_dates,3,3,None,None,PASS
feed_info,7,7,None,None,PASS
routes,9,9,None,None,PASS
shapes,5,5,None,None,PASS
stop_times,9,9,None,None,PASS
stops,11,11,None,None,PASS
translations,6,6,None,None,PASS
trips,8,8,None,None,PASS



INFERRED DATA TYPES


dataset,column_name,spark_data_type
agency,agency_id,string
agency,agency_name,string
agency,agency_timezone,string
agency,agency_url,string
calendar,end_date,int
calendar,friday,int
calendar,monday,int
calendar,saturday,int
calendar,service_id,int
calendar,start_date,int



CRITICAL IDENTIFIER NULL CHECK


...


PRIMARY IDENTIFIER DUPLICATE CHECK


dataset,identifier_column,duplicate_detected,status
routes,route_id,0,PASS
stops,stop_id,0,PASS
trips,trip_id,0,PASS



GTFS RELATIONSHIP VALIDATION


relationship,orphan_detected,status
routes.agency_id -> agency.agency_id,0,PASS
trips.route_id -> routes.route_id,0,PASS
stop_times.trip_id -> trips.trip_id,0,PASS
stop_times.stop_id -> stops.stop_id,0,PASS
trips.shape_id -> shapes.shape_id,0,PASS



GTFS VALIDATION SUMMARY
Schema issues: 0
Identifier issues: 0
Duplicate-key issues: 0
Relationship issues: 0
Total validation issues: 0

GTFS DATA QUALITY VALIDATION STATUS: PASS


In [0]:

print("=" * 70)
print("GTFS SILVER TRANSFORMATION")
print("=" * 70)

agency_silver = (
    agency
    .select(
        trim(col("agency_id").cast("string")).alias("agency_id"),
        trim(col("agency_name")).alias("agency_name"),
        trim(col("agency_url")).alias("agency_url"),
        trim(col("agency_timezone")).alias("agency_timezone")
    )
    .dropDuplicates(["agency_id"])
)

agency_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_agency_silver")

print("Created: workspace.default.gtfs_agency_silver")
print(f"Rows: {agency_silver.count():,}")

display(agency_silver)

GTFS SILVER TRANSFORMATION
Created: workspace.default.gtfs_agency_silver
Rows: 104


agency_id,agency_name,agency_url,agency_timezone
1024,Tralee People's Bus Service,http://www.transportforireland.ie,Europe/London
1030,McGonagle Bus and Coach Hire,https://www.mcgonaglebushire.com/,Europe/London
1134,Reilly Bus Hire,https://www.transportforireland.ie,Europe/London
1135,Joe Moroney Coach Hire Ltd,https://www.transportforireland.ie,Europe/London
616,Michael Moran,https://www.transportforireland.ie,Europe/London
967,Bus Feda,https://busfeda.ie/,Europe/London
Carolan,Balbriggan Express,http://www.carolancoachhire.ie/,Europe/London
LLTY,TFI Local Link Tipperary,https://locallinktipperary.ie/,Europe/London
LLWD,TFI Local Link Waterford,https://www.locallinkwaterford.ie/,Europe/London
NTLK,"Nitelink, Dublin Bus",https://www.dublinbus.ie/,Europe/London


In [0]:

print("=" * 70)
print("GTFS ROUTES SILVER TRANSFORMATION")
print("=" * 70)

routes_silver = (
    routes
    .select(
        trim(col("route_id").cast("string")).alias("route_id"),
        trim(col("agency_id").cast("string")).alias("agency_id"),
        trim(col("route_short_name")).alias("route_short_name"),
        trim(col("route_long_name")).alias("route_long_name"),
        trim(col("route_desc")).alias("route_desc"),
        col("route_type").cast("int").alias("route_type"),
        trim(col("route_url")).alias("route_url"),
        trim(col("route_color")).alias("route_color"),
        trim(col("route_text_color")).alias("route_text_color")
    )
    .withColumn(
        "transport_mode",
        when(col("route_type") == 0, "Luas")
        .when(col("route_type") == 2, "Rail")
        .when(col("route_type") == 3, "Bus")
        .otherwise("Other")
    )
    .dropDuplicates(["route_id"])
)

routes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_routes_silver")

print("Created: workspace.default.gtfs_routes_silver")
print(f"Rows: {routes_silver.count():,}")

display(
    routes_silver.select(
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name",
        "route_type",
        "transport_mode"
    )
)

GTFS ROUTES SILVER TRANSFORMATION
Created: workspace.default.gtfs_routes_silver
Rows: 805


route_id,agency_id,route_short_name,route_long_name,route_type,transport_mode
03C 120A e a,03C,120A,Dublin - Edenderry,3,Bus
03C 120C e a,03C,120C,Enfield - Tullamore,3,Bus
03C 130 e a,03C,130,Dublin - Athy,3,Bus
1 122 c a,1,122,Drimnagh - Ashington,3,Bus
1 130 c a,1,130,Talbot Street - Castle Avenue,3,Bus
1 37 c a,1,37,Blanchardstown Shopping Centre - Wilton Terrace,3,Bus
1 68 c a,1,68,Newcastle / Greenogue Business Pk - Hawkins St,3,Bus
1 70 c a,1,70,Dunboyne - Burlington Rd,3,Bus
1 80 a,1,80,Liffey Valley SC - Palmerston Park,3,Bus
1 E2 a,1,E2,Dun Laoghaire - Harristown,3,Bus


In [0]:

# STEP 5C — STOPS SILVER TRANSFORMATION

# Reuse stops data loaded in Step 3
stops_raw = gtfs_dfs["stops"]

# Clean and transform the NTA stops dataset
stops_silver = (
    stops_raw
    .select(
        trim(col("stop_id").cast("string")).alias("stop_id"),
        trim(col("stop_code").cast("string")).alias("stop_code"),
        trim(col("stop_name")).alias("stop_name"),
        trim(col("stop_desc")).alias("stop_desc"),

        col("stop_lat")
            .cast("double")
            .alias("stop_lat"),

        col("stop_lon")
            .cast("double")
            .alias("stop_lon"),

        col("location_type")
            .cast("int")
            .alias("location_type"),

        trim(col("parent_station").cast("string")).alias("parent_station"),
        trim(col("stop_url")).alias("stop_url"),

        col("wheelchair_boarding")
            .cast("int")
            .alias("wheelchair_boarding")
    )
    .dropDuplicates(["stop_id"])
)

# Save as Silver Delta table
stops_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_stops_silver")

print("=" * 70)
print("GTFS STOPS SILVER TRANSFORMATION")
print("=" * 70)

print("Created: workspace.default.gtfs_stops_silver")
print(f"Rows: {stops_silver.count():,}")

display(
    stops_silver.select(
        "stop_id",
        "stop_code",
        "stop_name",
        "stop_lat",
        "stop_lon",
        "location_type",
        "parent_station"
    )
)

GTFS STOPS SILVER TRANSFORMATION
Created: workspace.default.gtfs_stops_silver
Rows: 14,085


stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station
7040B107131,107131,Cullaville,54.0620448123748,-6.64352516920916,null,null
821000005,12324,Carlow Inst,52.8191,-6.94496,null,null
8210CO11096,110961,Hanover Road,52.831403,-6.929143,null,null
8210CO11100,111001,Deerpark Rnbt,52.845026,-6.910029,null,null
8210CO11110,111101,Burrindale,52.84195,-6.902001,null,null
8210CO11114,111141,Green Lane,52.84178,-6.916849,null,null
8210CO11119,111191,SETU Green Road,52.825738,-6.933882,null,null
8210CO11132,111321,Pollerton Road,52.836836,-6.918723,null,null
8210CO11157,111881,Sleaty Street,52.839859,-6.938004,null,null
8210LL10447,14523,Glynn,52.50274,-6.90664,null,null


In [0]:

# STEP 5D — TRIPS SILVER TRANSFORMATION

# Reuse trips data loaded in Step 3
trips_raw = gtfs_dfs["trips"]

# Clean and transform
trips_silver = (
    trips_raw
    .select(
        trim(col("route_id").cast("string")).alias("route_id"),
        trim(col("service_id").cast("string")).alias("service_id"),
        trim(col("trip_id").cast("string")).alias("trip_id"),
        trim(col("trip_headsign")).alias("trip_headsign"),

        col("direction_id")
            .cast("int")
            .alias("direction_id"),

        trim(col("block_id").cast("string")).alias("block_id"),
        trim(col("shape_id").cast("string")).alias("shape_id"),
        trim(col("trip_short_name")).alias("trip_short_name")
    )
    .dropDuplicates(["trip_id"])
)

# Save as Silver Delta table
trips_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_trips_silver")

print("=" * 70)
print("GTFS TRIPS SILVER TRANSFORMATION")
print("=" * 70)

print("Created: workspace.default.gtfs_trips_silver")
print(f"Rows: {trips_silver.count():,}")

display(
    trips_silver.select(
        "trip_id",
        "route_id",
        "service_id",
        "trip_headsign",
        "direction_id",
        "shape_id"
    )
)

GTFS TRIPS SILVER TRANSFORMATION
Created: workspace.default.gtfs_trips_silver
Rows: 199,547


trip_id,route_id,service_id,trip_headsign,direction_id,shape_id
4390_10,1011 930 a,1,Hawkswood,1,4390_3
4390_5,1011 930 a,2,Cavan Hospital,0,4390_1
4390_8,1011 930 a,2,Hawkswood,1,4390_3
4689_1,969 MB01 a,1,Moate Business School,0,4689_1
4869_5,640 AI02 b,1,Roscommon,1,4869_3
4988_4,JJBK 871 b,8,Urlingford,1,4988_2
5059_5,1059 920 b,2,Dublin Road,0,5059_2
5059_6,1059 920 b,1,Dublin Road,0,5059_2
5326_7,875 TT02,1,Tralee Bus Station,1,5326_5
5418_1,967 964 b,9,Croithlí,0,5418_8


In [0]:

# STEP 5E — STOP TIMES SILVER TRANSFORMATION

# Reuse stop_times data loaded in Step 3
stop_times_raw = gtfs_dfs["stop_times"]

# Clean and transform
stop_times_silver = (
    stop_times_raw
    .select(
        trim(col("trip_id").cast("string")).alias("trip_id"),
        trim(col("arrival_time")).alias("arrival_time"),
        trim(col("departure_time")).alias("departure_time"),

        trim(col("stop_id").cast("string")).alias("stop_id"),

        col("stop_sequence")
            .cast("int")
            .alias("stop_sequence"),

        trim(col("stop_headsign")).alias("stop_headsign"),

        col("pickup_type")
            .cast("int")
            .alias("pickup_type"),

        col("drop_off_type")
            .cast("int")
            .alias("drop_off_type"),

        trim(col("timepoint")).alias("timepoint")
    )
    .dropDuplicates(["trip_id", "stop_sequence"])
)

# Save as Silver Delta table
stop_times_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_stop_times_silver")

print("=" * 70)
print("GTFS STOP TIMES SILVER TRANSFORMATION")
print("=" * 70)

print("Created: workspace.default.gtfs_stop_times_silver")
print(f"Rows: {stop_times_silver.count():,}")

display(
    stop_times_silver.select(
        "trip_id",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "departure_time",
        "pickup_type",
        "drop_off_type"
    )
)

GTFS STOP TIMES SILVER TRANSFORMATION
Created: workspace.default.gtfs_stop_times_silver
Rows: 7,105,329


trip_id,stop_id,stop_sequence,arrival_time,departure_time,pickup_type,drop_off_type
5059_16,8470PB000872,6,08:05:00,08:05:00,0,0
5059_31,8470B5560301,7,19:25:00,19:25:00,0,0
5059_9,8470PB000648,11,16:47:00,16:47:00,0,0
5424_30,8220000150,20,12:30:00,12:30:00,1,0
5424_57,8220000150,1,14:00:00,14:00:00,0,1
5424_7,8300B1524401,15,16:00:00,16:00:00,0,0
5425_25,8490B582171,1,26:00:00,26:00:00,0,1
5432_5,8470PB14705,12,11:40:00,11:40:00,0,0
5453_19,853000081,6,05:20:00,05:20:00,0,1
5453_53,853000124,2,16:00:00,16:00:00,0,1


In [0]:

# STEP 5F — GTFS CALENDAR SILVER TRANSFORMATION

# Reuse calendar data loaded in Step 3
calendar_raw = gtfs_dfs["calendar"]

# Transform into Silver layer
calendar_silver = (
    calendar_raw
    .select(
        trim(col("service_id").cast("string")).alias("service_id"),

        col("monday").cast("int").alias("monday"),
        col("tuesday").cast("int").alias("tuesday"),
        col("wednesday").cast("int").alias("wednesday"),
        col("thursday").cast("int").alias("thursday"),
        col("friday").cast("int").alias("friday"),
        col("saturday").cast("int").alias("saturday"),
        col("sunday").cast("int").alias("sunday"),

        to_date(
            col("start_date").cast("string"),
            "yyyyMMdd"
        ).alias("start_date"),

        to_date(
            col("end_date").cast("string"),
            "yyyyMMdd"
        ).alias("end_date")
    )
    .dropDuplicates(["service_id"])
)

# Save as Delta Silver table
calendar_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_calendar_silver")


print("=" * 70)
print("GTFS CALENDAR SILVER TRANSFORMATION")
print("=" * 70)

print("\nCreated: workspace.default.gtfs_calendar_silver")
print(f"Rows: {calendar_silver.count():,}")

display(
    calendar_silver.select(
        "service_id",
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday",
        "saturday",
        "sunday",
        "start_date",
        "end_date"
    )
)

GTFS CALENDAR SILVER TRANSFORMATION

Created: workspace.default.gtfs_calendar_silver
Rows: 285


service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
56,1,1,1,1,1,1,0,2026-09-29,2026-10-17
73,0,0,0,0,0,0,1,2026-10-04,2026-10-26
110,1,1,1,1,1,0,0,2026-09-29,2027-07-09
120,0,1,0,0,0,0,0,2026-09-29,2027-09-28
146,0,0,0,0,0,1,0,2026-10-03,2026-10-03
149,0,0,0,1,1,0,0,2026-10-01,2027-09-30
210,1,1,1,1,1,1,0,2026-10-21,2027-09-30
226,0,0,0,0,0,0,1,2026-10-04,2027-05-23
245,0,1,0,0,0,0,0,2026-09-29,2027-09-28
250,0,0,0,0,0,0,1,2026-10-04,2027-09-26


In [0]:

from pyspark.sql.functions import col, trim, to_date

# STEP 5G — GTFS CALENDAR DATES SILVER TRANSFORMATION

# Use the GTFS file location already verified in Databricks
calendar_dates_path = (
    "/Volumes/workspace/default/"
    "dublin_transport_gtfs/raw_gtfs/calendar_dates.txt"
)

# Load raw calendar dates
calendar_dates_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(calendar_dates_path)
)

# Transform to Silver
calendar_dates_silver = (
    calendar_dates_raw
    .select(
        trim(col("service_id")).alias("service_id"),
        to_date(col("date"), "yyyyMMdd").alias("date"),
        col("exception_type").cast("int").alias("exception_type")
    )
    .dropDuplicates(["service_id", "date"])
)

# Save Delta table
calendar_dates_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gtfs_calendar_dates_silver")

print("=" * 70)
print("GTFS CALENDAR DATES SILVER TRANSFORMATION")
print("=" * 70)
print("Created: workspace.default.gtfs_calendar_dates_silver")
print(f"Rows: {calendar_dates_silver.count():,}")

display(calendar_dates_silver)


GTFS CALENDAR DATES SILVER TRANSFORMATION
Created: workspace.default.gtfs_calendar_dates_silver
Rows: 1,811


service_id,date,exception_type
24,2027-01-01,1
37,2026-10-26,2
40,2027-09-06,1
40,2027-09-14,1
41,2027-08-31,1
41,2027-09-09,1
47,2027-03-17,2
75,2027-03-29,2
77,2027-07-07,1
77,2027-08-11,1


In [0]:

from pyspark.sql.functions import col

# STEP 5H — GTFS CALENDAR DATES DATA QUALITY VALIDATION

calendar_dates = spark.table(
    "workspace.default.gtfs_calendar_dates_silver"
)

total_rows = calendar_dates.count()

null_service_ids = calendar_dates.filter(
    col("service_id").isNull()
).count()

null_dates = calendar_dates.filter(
    col("date").isNull()
).count()

invalid_exceptions = calendar_dates.filter(
    col("exception_type").isNull() |
    (~col("exception_type").isin(1, 2))
).count()

duplicate_records = (
    total_rows -
    calendar_dates.select("service_id", "date").distinct().count()
)

print("=" * 65)
print("STEP 5H — CALENDAR DATES DATA QUALITY VALIDATION")
print("=" * 65)

print(f"Total records: {total_rows:,}")
print(f"Null service IDs: {null_service_ids:,}")
print(f"Null dates: {null_dates:,}")
print(f"Invalid exception types: {invalid_exceptions:,}")
print(f"Duplicate service/date combinations: {duplicate_records:,}")

print("\nException type distribution:")
display(
    calendar_dates.groupBy("exception_type").count()
    .orderBy("exception_type")
)


STEP 5H — CALENDAR DATES DATA QUALITY VALIDATION
Total records: 1,811
Null service IDs: 0
Null dates: 0
Invalid exception types: 0
Duplicate service/date combinations: 0

Exception type distribution:


exception_type,count
1,701
2,1110


In [0]:

# STEP 5I — GTFS REFERENTIAL INTEGRITY VALIDATION

from pyspark.sql.functions import col

routes = spark.table("workspace.default.gtfs_routes_silver")
trips = spark.table("workspace.default.gtfs_trips_silver")
stops = spark.table("workspace.default.gtfs_stops_silver")
stop_times = spark.table("workspace.default.gtfs_stop_times_silver")
calendar = spark.table("workspace.default.gtfs_calendar_silver")
calendar_dates = spark.table("workspace.default.gtfs_calendar_dates_silver")

# 1. Trips referencing routes that do not exist
missing_routes = (
    trips.select("route_id").distinct()
    .join(
        routes.select("route_id").distinct(),
        "route_id",
        "left_anti"
    )
)

# 2. Stop times referencing trips that do not exist
missing_trips = (
    stop_times.select("trip_id").distinct()
    .join(
        trips.select("trip_id").distinct(),
        "trip_id",
        "left_anti"
    )
)

# 3. Stop times referencing stops that do not exist
missing_stops = (
    stop_times.select("stop_id").distinct()
    .join(
        stops.select("stop_id").distinct(),
        "stop_id",
        "left_anti"
    )
)

# 4. Trips referencing service IDs absent from both calendars
valid_services = (
    calendar.select("service_id")
    .union(calendar_dates.select("service_id"))
    .distinct()
)

missing_services = (
    trips.select("service_id").distinct()
    .join(valid_services, "service_id", "left_anti")
)

print("=" * 65)
print("STEP 5I — GTFS REFERENTIAL INTEGRITY VALIDATION")
print("=" * 65)

print(f"Missing route references: {missing_routes.count():,}")
print(f"Missing trip references: {missing_trips.count():,}")
print(f"Missing stop references: {missing_stops.count():,}")
print(f"Missing service references: {missing_services.count():,}")


STEP 5I — GTFS REFERENTIAL INTEGRITY VALIDATION
Missing route references: 0
Missing trip references: 0
Missing stop references: 0
Missing service references: 0


In [0]:

# STEP 5J — GTFS SOURCE-TO-SILVER ROW RECONCILIATION

gtfs_path = (
    "/Volumes/workspace/default/"
    "dublin_transport_gtfs/raw_gtfs/"
)

tables = [
    "agency",
    "routes",
    "stops",
    "trips",
    "stop_times",
    "calendar",
    "calendar_dates"
]

results = []

for table_name in tables:

    raw_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(f"{gtfs_path}{table_name}.txt")
    )

    silver_df = spark.table(
        f"workspace.default.gtfs_{table_name}_silver"
    )

    raw_count = raw_df.count()
    silver_count = silver_df.count()

    results.append((
        table_name,
        raw_count,
        silver_count,
        raw_count - silver_count
    ))

reconciliation_df = spark.createDataFrame(
    results,
    [
        "table_name",
        "raw_rows",
        "silver_rows",
        "row_difference"
    ]
)

display(reconciliation_df)


table_name,raw_rows,silver_rows,row_difference
agency,104,104,0
routes,805,805,0
stops,14085,14085,0
trips,199547,199547,0
stop_times,7105329,7105329,0
calendar,285,285,0
calendar_dates,1811,1811,0


In [0]:
# STEP 6A — Fresh passenger source and GTFS Silver references
from pyspark.sql import functions as F
from datetime import date

CATALOG = "workspace.default"
journeys = spark.table(f"{CATALOG}.passenger_journeys_silver")
trips = spark.table(f"{CATALOG}.gtfs_trips_silver").select("trip_id", "route_id", "service_id")
routes = spark.table(f"{CATALOG}.gtfs_routes_silver").select("route_id").distinct()
stops = spark.table(f"{CATALOG}.gtfs_stops_silver").select("stop_id").distinct()
stop_times = spark.table(f"{CATALOG}.gtfs_stop_times_silver").select("trip_id", "stop_id", "stop_sequence").distinct()
calendar = spark.table(f"{CATALOG}.gtfs_calendar_silver")
calendar_dates = spark.table(f"{CATALOG}.gtfs_calendar_dates_silver")

source = spark.table(f"{CATALOG}.passenger_journeys")
assert source.count() == journeys.count(), "Bronze/source and Silver passenger row counts differ"
summary = journeys.agg(F.count('*').alias('journeys'), F.min('journey_date').alias('first_date'), F.max('journey_date').alias('last_date')).first()
print('Passenger journeys:', summary['journeys'], '| date range:', summary['first_date'], 'to', summary['last_date'])
assert summary['journeys'] == 1000, 'Expected 1,000 regenerated synthetic journeys'
assert summary['first_date'] == date(2026, 9, 30) and summary['last_date'] == date(2026, 10, 30), 'Passenger tables are not refreshed to the October CSV; rerun Notebook 01'


Passenger journeys: 1000 | date range: 2026-09-30 to 2026-10-30


In [0]:
# STEP 6B–6D — Referential integrity, route matching and stop sequence checks
trip_keys = trips.select('trip_id').distinct()
checks = {
    'unknown_trip': journeys.join(trip_keys, 'trip_id', 'left_anti').count(),
    'unknown_route': journeys.join(routes, 'route_id', 'left_anti').count(),
    'unknown_origin_stop': journeys.join(stops, journeys.origin_stop_id == stops.stop_id, 'left_anti').count(),
    'unknown_destination_stop': journeys.join(stops, journeys.destination_stop_id == stops.stop_id, 'left_anti').count(),
}
trip_route = trips.select('trip_id', F.col('route_id').alias('gtfs_route_id'), 'service_id')
joined = journeys.join(trip_route, 'trip_id', 'left')
checks['trip_route_mismatch'] = joined.filter(F.col('gtfs_route_id').isNull() | (F.col('route_id') != F.col('gtfs_route_id'))).count()
origin_valid = journeys.alias('j').join(stop_times.alias('s'), (F.col('j.trip_id') == F.col('s.trip_id')) & (F.col('j.origin_stop_id') == F.col('s.stop_id')) & (F.col('j.origin_stop_sequence') == F.col('s.stop_sequence')), 'left_semi')
destination_valid = journeys.alias('j').join(stop_times.alias('s'), (F.col('j.trip_id') == F.col('s.trip_id')) & (F.col('j.destination_stop_id') == F.col('s.stop_id')) & (F.col('j.destination_stop_sequence') == F.col('s.stop_sequence')), 'left_semi')
checks['invalid_origin_trip_stop_sequence'] = journeys.count() - origin_valid.count()
checks['invalid_destination_trip_stop_sequence'] = journeys.count() - destination_valid.count()
checks['invalid_stop_order'] = journeys.filter(F.col('origin_stop_sequence') >= F.col('destination_stop_sequence')).count()
display(spark.createDataFrame([(k, int(v)) for k,v in checks.items()], ['check','invalid_journeys']))


check,invalid_journeys
unknown_trip,0
unknown_route,0
unknown_origin_stop,0
unknown_destination_stop,0
trip_route_mismatch,0
invalid_origin_trip_stop_sequence,0
invalid_destination_trip_stop_sequence,0
invalid_stop_order,0


In [0]:
# STEP 6E–6G — Service calendar coverage and date-range diagnostics
journey_services = joined.select('service_id').where(F.col('service_id').isNotNull()).distinct()
calendar_services = calendar.select('service_id').unionByName(calendar_dates.select('service_id')).distinct()
print('Journey service IDs:', journey_services.count())
print('Journey services without any GTFS calendar records:', journey_services.join(calendar_services, 'service_id', 'left_anti').count())
print('GTFS base calendar coverage:')
display(calendar.agg(F.min('start_date').alias('first_start'), F.max('end_date').alias('last_end'), F.countDistinct('service_id').alias('services')))
range_check = (joined.join(calendar.select('service_id','start_date','end_date'), 'service_id', 'left')
 .withColumn('date_range_status', F.when(F.col('service_id').isNull(),'MISSING_TRIP')
  .when(F.col('start_date').isNull(),'NO_BASE_CALENDAR')
  .when(F.col('journey_date') < F.col('start_date'),'BEFORE_SERVICE_START')
  .when(F.col('journey_date') > F.col('end_date'),'AFTER_SERVICE_END')
  .otherwise('WITHIN_BASE_RANGE')))
display(range_check.groupBy('date_range_status').count().orderBy('date_range_status'))


Journey service IDs: 53
Journey services without any GTFS calendar records: 0
GTFS base calendar coverage:


first_start,last_end,services
2026-09-29,2027-09-30,285


date_range_status,count
WITHIN_BASE_RANGE,1000


In [0]:
# STEP 6H — Construct actual operating dates from base calendar and exceptions
# Restrict expansion to the synthetic journey window, rather than expanding all GTFS dates.
window_start = F.lit('2026-09-30').cast('date')
window_end = F.lit('2026-10-30').cast('date')
relevant_calendar = calendar.join(journey_services, 'service_id', 'inner')
expanded = (relevant_calendar
 .withColumn('from_date', F.greatest(F.col('start_date'), window_start))
 .withColumn('to_date', F.least(F.col('end_date'), window_end))
 .where(F.col('from_date') <= F.col('to_date'))
 .withColumn('service_date', F.explode(F.sequence('from_date','to_date', F.expr('INTERVAL 1 DAY')))))
# ISO dayofweek: Monday=1 ... Sunday=7
weekday_flag = F.element_at(F.array(*[F.col(d) for d in ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']]), F.pmod(F.dayofweek('service_date') + F.lit(5), F.lit(7)) + F.lit(1))
base_operating = expanded.filter(weekday_flag == 1).select('service_id','service_date')
exceptions = (calendar_dates.join(journey_services,'service_id','inner')
 .select('service_id',F.col('date').alias('service_date'),'exception_type')
 .where(F.col('service_date').between(window_start,window_end)))
additions = exceptions.where(F.col('exception_type') == 1).select('service_id','service_date')
removals = exceptions.where(F.col('exception_type') == 2).select('service_id','service_date')
valid_service_dates = (base_operating.unionByName(additions).distinct()
 .join(removals,['service_id','service_date'],'left_anti'))
print('Actual valid service/date pairs in journey window:',valid_service_dates.count())
display(valid_service_dates.agg(F.min('service_date').alias('first_operating_date'),F.max('service_date').alias('last_operating_date')))


Actual valid service/date pairs in journey window: 423


first_operating_date,last_operating_date
2026-09-30,2026-10-30


In [0]:
# STEP 6I–6J — Validate each passenger journey against its trip's operating dates
valid_pairs = valid_service_dates.select(F.col('service_id').alias('valid_service_id'), F.col('service_date').alias('valid_service_date'))
journey_validation = (joined.alias('j').join(valid_pairs.alias('v'),
 (F.col('j.service_id') == F.col('v.valid_service_id')) & (F.to_date(F.col('j.journey_date')) == F.col('v.valid_service_date')), 'left')
 .withColumn('operating_date_status',
  F.when(F.col('service_id').isNull(),'MISSING_TRIP_OR_SERVICE')
   .when(F.col('valid_service_date').isNotNull(),'VALID_OPERATING_DATE')
   .otherwise('INVALID_OPERATING_DATE'))
 .drop('valid_service_id','valid_service_date'))
print('Validated journeys:', journey_validation.count())
display(journey_validation.groupBy('operating_date_status').count().orderBy('operating_date_status'))
display(journey_validation.where(F.col('operating_date_status') != 'VALID_OPERATING_DATE')
 .select('journey_id','journey_date','trip_id','service_id','operating_date_status').limit(20))


Validated journeys: 1000


operating_date_status,count
VALID_OPERATING_DATE,1000


journey_id,journey_date,trip_id,service_id,operating_date_status


In [0]:
# STEP 6K — Diagnose valid dates for services linked to invalid journeys (do not alter the synthetic records)
invalid_services = journey_validation.where(F.col('operating_date_status') != 'VALID_OPERATING_DATE').select('service_id').where(F.col('service_id').isNotNull()).distinct()
replacement_availability = (invalid_services.join(valid_service_dates,'service_id','left')
 .groupBy('service_id').agg(F.count('service_date').alias('valid_dates_in_window'),
 F.min('service_date').alias('first_valid_date'),F.max('service_date').alias('last_valid_date')))
display(replacement_availability.orderBy('valid_dates_in_window','service_id').limit(30))
print('Validation is diagnostic: no journey dates were rewritten.')


service_id,valid_dates_in_window,first_valid_date,last_valid_date


Validation is diagnostic: no journey dates were rewritten.
